# Lesson 3: Health check: 41 status queries

Inspect each subsystem section by section, beyond the short snapshot in
[lesson 2](02_quick_status.ipynb). Each section has its own query cell and stores
byte payloads in `health`. This is an inspection record, not an automatic
pass/fail assessment or authorization to enable the laser.

Run cells top to bottom with only one serial owner. Restart the kernel after a
package update. The instrument prompt must already be enabled. Each `read()`
consumes the complete reply CR/LF and enabled trailing prompt before another request.

`read()` returns payload `bytes` with echo, CR/LF framing, recognized leading
firmware notices and prompt removed. It does not convert the payload to numbers,
booleans or fault codes. Bytes returned by serial reads, including notices and incomplete replies returned
at a timeout, remain in `laser.last_transaction["response"]`.

**Queries only:** no settings, fault resets, heater changes or optimization commands.
If a cell raises an error, stop, inspect the evidence cell and run Disconnect.
Do not retry an uncertain exchange.

## 1. Connection settings

Use the port identified in lesson 0 and confirm model and baud on the instrument.
No fault-clear text is required to collect raw replies. Interpret their meaning
with the operator and the [manual](../verdi.manual_v5.pdf).

With `prompt_enabled=True`, the controller reads the reply through CR/LF, then
consumes the enabled trailing prompt before sending another instruction. The
instrument prompt must already be enabled; construction sends no echo or prompt
changes. Matching instruction echoes and recognized leading notices are removed
from `read()` payloads while their received bytes remain in transaction evidence.

`TIMEOUT_S` configures serial write and receive waits. A reply line, each leading
notice and the trailing prompt use separate `read_until()` calls with that
configured timeout; there is no combined transaction deadline. No extra pause
is inserted between queries. Restart the kernel after a package update.

Saved outputs and execution counts are retained from earlier hardware runs;
they have not been regenerated for these revised cells.

In [1]:
PORT = "COM5"
MODEL = "V2"  # V2, V5 or V6: match the actual laser
BAUDRATE = 19200  # Match the front panel
TIMEOUT_S = 2.0  # Serial write and receive timeout

## 2. Create a read-only controller

Construction opens nothing. Writes remain disabled.

In [2]:
from coherent_verdi import VerdiController

if PORT == "<PORT>":
    raise ValueError("Edit the connection settings in section 1 first.")
laser = VerdiController(
    PORT,
    model=MODEL,
    baudrate=BAUDRATE,
    timeout_s=TIMEOUT_S,
    prompt_enabled=True,
)

## 3. Open the serial port

Run this cell by itself and inspect the result before starting serial communication.
`connect()` opens the local serial port and returns `True` on success. Failures
raise an exception. It sends no instrument instructions. A successful open does
not establish that the laser is responding or ready.

Close any other serial owner first. If opening fails, stop here.

`override=True` closes an older controller connection to the same port in this
Python process before opening the new one. It cannot take ownership from another
kernel/application or a raw `ser` handle; release those explicitly. A failed
controller still requires a fresh controller object. Reopening does not establish
instrument synchronization or repair corrupted replies.

In [3]:
connected = laser.connect(override=True)
print("Serial port open:", connected)

Serial port open: True


## 4. Start serial communication

Only after inspecting the connection result, run the following cells separately.
Buffer maintenance and instrument queries belong here, outside the connection
section. An explicit purge discards host queues; it does not reset the device's
parser. No delay or purge is claimed to prove synchronization. Each query consumes
the complete reply CR/LF and enabled trailing prompt before another can be sent.

In [4]:
def require_query_reply(query, reply):
    # Inspect payload byte markers without interpreting values; wire evidence is retained.
    if any(marker in reply for marker in (b"Query Error:", b"Command Error:", b"RANGE ERROR:")):
        raise RuntimeError(
            f"Device rejected {query!r}: {reply!r}. Stop the survey and inspect the link. "
            "See the optional version diagnostic."
        )


health = {}

In [5]:
laser.purge_buffers()  # Explicit host TX/RX cleanup before the first request.
version_reply = laser.read(b"?SV")
print(repr(version_reply))
require_query_reply(b"?SV", version_reply)

b'6.05'


### Optional: one version diagnostic on the same connection

If the version query above returned a complete device-error reply such as
`b" Query Error:?*?SV\r\nVerdi V-2>"`, stop the survey and run this optional
cell once after changing its flag to `True`. It saves and prints the previous
transaction, sends one read-only `?SV` query on the **same connection**, then
saves and prints the new transaction. No purge, reconnect, encoding, decoding
or automatic retry loop is used.

This diagnostic is disabled by default and is allowed only after a complete
rejected version query. Do not use it after a timeout, incomplete reply or failure
of another query. Inspect both records before continuing; a successful second
reply does not prove the underlying issue is fixed.

In [6]:
RUN_VERSION_DIAGNOSTIC = False  # Change to True only for this deliberate diagnostic.

if RUN_VERSION_DIAGNOSTIC:
    previous_transaction = laser.last_transaction
    print("Previous transaction:", previous_transaction)
    if (
        laser.failed
        or previous_transaction is None
        or previous_transaction["request"] != b"?SV\r\n"
        or previous_transaction["outcome"] != "received"
        or not any(
            marker in previous_transaction["response"]
            for marker in (
                b"Query Error:",
                b"Command Error:",
                b"RANGE ERROR:",
            )
        )
    ):
        raise RuntimeError("Diagnostic requires a complete rejected ?SV reply on this connection.")
    try:
        diagnostic_reply = laser.read(b"?SV")
        print("Diagnostic reply:", repr(diagnostic_reply))
    finally:
        diagnostic_transaction = laser.last_transaction
        print("Diagnostic transaction:", diagnostic_transaction)
    require_query_reply(b"?SV", diagnostic_reply)

## 5. Output and operating state

Compare reported output with its setpoint and the operating state. These are sequential samples, not a simultaneous measurement.

In [7]:
queries = {
    b"?P": "Actual output power",
    b"?SP": "Power setpoint",
    b"?L": "Laser state",
    b"?S": "Shutter state",
    b"?K": "Keyswitch state",
    b"?M": "Regulation mode",
    b"?LRS": "Light regulation servo status",
    b"?C": "Current",
}

health["Output and operating state"] = {}
for query, label in queries.items():
    reply = laser.read(query)
    health["Output and operating state"][query] = reply
    print(f"{label:36s}: {reply!r}")
    require_query_reply(query, reply)

Actual output power                 : b'0.010'
Power setpoint                      : b'0.0100'
Laser state                         : b'0'
Shutter state                       : b'0'
Keyswitch state                     : b'0'
Regulation mode                     : b'1'
Light regulation servo status       : b'2'
Current                             : b'0.00'


## 6. Faults and identification

Preserve active faults and history before considering any action. Look up every code, including unknown ones, in the [fault reference](../docs/PROTOCOL.md#fault-code-reference). No faults are cleared. The version is queried again here as part of the full survey.

In [8]:
queries = {
    b"?F": "Active faults",
    b"?FH": "Fault history",
    b"?SV": "Software version",
    b"?B": "Baud rate",
}

health["Faults and identification"] = {}
for query, label in queries.items():
    reply = laser.read(query)
    health["Faults and identification"][query] = reply
    print(f"{label:36s}: {reply!r}")
    require_query_reply(query, reply)

Active faults                       : b'SYSTEM OK'
Fault history                       : b'SYSTEM OK'
Software version                    : b'6.05'
Baud rate                           : b'19200'


## 7. Operating hours

Record these counters for maintenance comparisons. They are instrument-reported operating hours, not a remaining-life estimate.

In [9]:
queries = {
    b"?HH": "Laser head operating hours",
    b"?PSH": "Power supply operating hours",
    b"?D1H": "Diode 1 operating hours",
}

health["Operating hours"] = {}
for query, label in queries.items():
    reply = laser.read(query)
    health["Operating hours"][query] = reply
    print(f"{label:36s}: {reply!r}")
    require_query_reply(query, reply)

Laser head operating hours          : b'2694.91'
Power supply operating hours        : b'2694.91'
Diode 1 operating hours             : b'2390123.54'


## 8. Laser head

Inspect the laser head baseplate temperature alongside the cooling conditions. Internal crystal temperatures are separate readings below.

In [10]:
queries = {
    b"?BT": "Baseplate temperature",
}

health["Laser head"] = {}
for query, label in queries.items():
    reply = laser.read(query)
    health["Laser head"][query] = reply
    print(f"{label:36s}: {reply!r}")
    require_query_reply(query, reply)

Baseplate temperature               : b'25.23'


## 9. LBO

Compare measured temperature with the setpoint and servo status. Heater state and optimizer availability are queries only; this cell does not change the heater or start optimization.

In [11]:
queries = {
    b"?LBOT": "LBO measured temperature",
    b"?LBOST": "LBO temperature setpoint",
    b"?LBOSS": "LBO temperature servo status",
    b"?LBOD": "LBO temperature servo drive",
    b"?LBOH": "LBO heater state",
    b"?LBOOS": "LBO optimizer availability",
}

health["LBO"] = {}
for query, label in queries.items():
    reply = laser.read(query)
    health["LBO"][query] = reply
    print(f"{label:36s}: {reply!r}")
    require_query_reply(query, reply)

LBO measured temperature            : b'149.24'
LBO temperature setpoint            : b'149.20'
LBO temperature servo status        : b'1'
LBO temperature servo drive         : b'6621.09'
LBO heater state                    : b'1'
LBO optimizer availability          : b'0'


## 10. Etalon

Inspect measured temperature, setpoint and servo status together. Keep the servo-drive reply as reported; no units or conversion are guessed.

In [12]:
queries = {
    b"?ET": "Etalon measured temperature",
    b"?EST": "Etalon temperature setpoint",
    b"?ESS": "Etalon temperature servo status",
    b"?ED": "Etalon temperature servo drive",
}

health["Etalon"] = {}
for query, label in queries.items():
    reply = laser.read(query)
    health["Etalon"][query] = reply
    print(f"{label:36s}: {reply!r}")
    require_query_reply(query, reply)

Etalon measured temperature         : b'70.07'
Etalon temperature setpoint         : b'70.00'
Etalon temperature servo status     : b'1'
Etalon temperature servo drive      : b'4030.29'


## 11. Vanadate

Compare measured temperature with this unit's setpoint and servo status, rather than assuming a universal operating temperature.

In [13]:
queries = {
    b"?VT": "Vanadate measured temperature",
    b"?VST": "Vanadate temperature setpoint",
    b"?VSS": "Vanadate temperature servo status",
    b"?VD": "Vanadate temperature servo drive",
}

health["Vanadate"] = {}
for query, label in queries.items():
    reply = laser.read(query)
    health["Vanadate"][query] = reply
    print(f"{label:36s}: {reply!r}")
    require_query_reply(query, reply)

Vanadate measured temperature       : b'34.99'
Vanadate temperature setpoint       : b'35.00'
Vanadate temperature servo status   : b'1'
Vanadate temperature servo drive    : b'1057.26'


## 12. Diode 1

Record the measured values, setpoints and diagnostic readings. Rated maximum current is not an instruction or an approved operating target.

In [14]:
queries = {
    b"?D1C": "Diode 1 current",
    b"?D1T": "Diode 1 measured temperature",
    b"?D1ST": "Diode 1 temperature setpoint",
    b"?D1SS": "Diode 1 temperature servo status",
    b"?D1TD": "Diode 1 temperature servo drive",
    b"?D1HST": "Diode 1 heatsink temperature",
    b"?D1PC": "Diode 1 photocell reading",
    b"?D1RCF": "Diode 1 rated current factor",
    b"?D1RCM": "Diode 1 rated maximum current",
    b"?D15V": "Diode 1 temperature reference voltage",
}

health["Diode 1"] = {}
for query, label in queries.items():
    reply = laser.read(query)
    health["Diode 1"][query] = reply
    print(f"{label:36s}: {reply!r}")
    require_query_reply(query, reply)

Diode 1 current                     : b'0.08'
Diode 1 measured temperature        : b'27.87'
Diode 1 temperature setpoint        : b'27.87'
Diode 1 temperature servo status    : b'1'
Diode 1 temperature servo drive     : b'180.67'
Diode 1 heatsink temperature        : b'23.81'
Diode 1 photocell reading           : b'0.01'
Diode 1 rated current factor        : b'0.02'
Diode 1 rated maximum current       : b'22.41'
Diode 1 temperature reference voltage: b'5.00'


## 13. Diode optimization and aging

Read aging-related information without starting optimization. `?DIOS` is deliberately skipped because this laser returned `Query Error:?DIOS`. `?ACAD` is still queried; keep its payload, including compound values.


In [15]:
queries = {
    b"?ACAD": "Average current and delta",
}

health["Diode optimization and aging"] = {}
for query, label in queries.items():
    reply = laser.read(query)
    health["Diode optimization and aging"][query] = reply
    print(f"{label:36s}: {reply!r}")
    require_query_reply(query, reply)

Average current and delta           : b'0.080.0'


## 14. Inspect an error

After a failed cell, run this cell to inspect the last transaction without sending
anything. Previously completed replies remain in `health`. An incomplete reply
or missing prompt stops the session; reconnecting alone does not establish alignment.
Raw device error replies remain visible as bytes and are not evidence of health.

A `Query Error:` or `Command Error:` is an error even when CR/LF and prompt
were received correctly. This notebook stops on those byte markers. Host buffer
purges do not repair a malformed instruction inside the laser. After a failure,
inspect the transaction and disconnect; do not proceed to other sections.

In [16]:
print(laser.last_transaction)

{'instruction': b'?ACAD', 'request': b'?ACAD\r\n', 'response': b'0.080.0\r\nVerdi V-2>', 'payload': b'0.080.0', 'outcome': 'received', 'error': None}


## 15. Disconnect

Release the port without changing laser state. No faults have been cleared.
Keep this survey for comparison with the front panel and later maintenance records.
Continue to [lesson 4: Pre-key-on readiness](04_check_readiness.ipynb) for
warm-up, fresh serial readiness checks and the physical checklist.

In [17]:
laser.disconnect()